# STAIR Baseline Reproduction & Benchmark Experiment

**Mục tiêu:** Tái tạo kết quả trong paper STAIR (AAAI 2025) và mở rộng đối chuẩn trên tập dữ liệu micro-video TikTok (ACM MM 2024 / DiffMM).  
**Môi trường:** Kaggle Notebook — GPU T4 / P100, 16 GB VRAM  
**Datasets:** Amazon2014Baby_550_MMRec, Amazon2014Sports_550_MMRec, Amazon2014Electronics_550_MMRec, và **TikTok** (Tri-modal: Vision, Text, Audio).  

> **[AUDIT FIX v2]** Chạy training bằng `--config configs/<dataset>.yaml` (không list flag rời)  
> để đảm bảo `monitors`, `which4best`, `eval-freq`, `mfiles`, `gamma` được nạp đúng từ yaml.  
> **[TIKTOK EXTENSION]** Thêm Cell 5c chạy thực nghiệm STAIR trên tập TikTok với cấu hình Tri-modal phù hợp kịch bản content-driven micro-video.  

**Lưu ý:** Do giới hạn GPU-hour Kaggle, mỗi dataset chạy 1 lần (seed=1). Kết quả được trích xuất và hiển thị tự động vào bảng tổng kết.


In [ ]:
# ================================================================
# CELL 1: Environment Setup
# - Clone STAIR-Enhanced (chứa mã nguồn STAIR, configs và dataset TikTok)
# - Pin freerec==0.8.5 (env.sh gốc tác giả)
# - Pin torchdata==0.7.1
# - PyG via official wheel index
# ================================================================
import os, shutil, subprocess, sys, time

os.chdir('/kaggle/working')
if os.path.exists('STAIR'):
    shutil.rmtree('STAIR')

REPO_URL = os.environ.get('REPO_URL', 'https://github.com/ThanhChuong12/STAIR-Enhanced.git')
print(f'Cloning repository from {REPO_URL}...')
subprocess.run(
    ['git', 'clone', '--depth', '1', REPO_URL, 'STAIR'],
    check=True
)

print('Installing freerec==0.8.5 & torchdata==0.7.1...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'freerec==0.8.5', 'torchdata==0.7.1', 'nvidia-ml-py'],
    check=True
)

import torch
TORCH_VER = torch.__version__.split('+')[0]
CUDA_TAG  = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f'Installing torch-geometric for torch={TORCH_VER}+{CUDA_TAG}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric',
     '-f', f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_TAG}.html'],
    check=True
)

print('\n[OK] Environment setup complete!')
print(f'   torch={torch.__version__}, cuda={torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')


In [ ]:
# ================================================================
# CELL 2: Data Preparation
# - DATA_ROOT = /kaggle/working/STAIR/data (khớp repo structure)
# - Hỗ trợ Amazon Baby, Sports, Electronics và TikTok
# ================================================================
import os, shutil

DATA_ROOT = '/kaggle/working/STAIR/data'
os.makedirs(DATA_ROOT, exist_ok=True)

DATASET_MAP = {
    'Amazon2014Baby_550_MMRec':
        '/kaggle/input/datasets/rainyle/stair-datasets-mmrec/Amazon2014Baby_550_MMRec',
    'Amazon2014Sports_550_MMRec':
        '/kaggle/input/datasets/rainyle/stair-datasets-mmrec/Amazon2014Sports_550_MMRec',
    'Amazon2014Electronics_550_MMRec':
        '/kaggle/input/datasets/rainyle/stair-datasets-mmrec/Amazon2014Electronics_550_MMRec',
}

REQUIRED_FILES = {
    'train.txt', 'valid.txt', 'test.txt',
    'textual_modality.pkl', 'visual_modality.pkl'
}

for ds_name, src in DATASET_MAP.items():
    dest = os.path.join(DATA_ROOT, ds_name)
    if os.path.exists(src):
        if os.path.exists(dest):
            shutil.rmtree(dest)
        shutil.copytree(src, dest)
    elif os.path.exists(dest):
        pass
    else:
        print(f'[Skip/Optional] {ds_name} source not found: {src}')
        continue
    present = set(os.listdir(dest))
    missing = REQUIRED_FILES - present
    if missing:
        print(f'[Warning] [{ds_name}] Missing: {missing}')
    else:
        sizes = {f: f'{os.path.getsize(os.path.join(dest, f))/1024/1024:.1f}MB' for f in sorted(present)}
        print(f'[OK] {ds_name}')
        for fname, sz in sizes.items():
            print(f'       {fname}: {sz}')

# Kiểm tra dataset TikTok (đã đóng gói sẵn trong repo STAIR-Enhanced/data/tiktok)
tiktok_dest = os.path.join(DATA_ROOT, 'tiktok')
if os.path.exists(tiktok_dest):
    tiktok_present = set(os.listdir(tiktok_dest))
    tiktok_req = {'train.txt', 'valid.txt', 'test.txt', 'visual_modality.pkl', 'textual_modality.pkl', 'audio_modality.pkl'}
    missing_tk = tiktok_req - tiktok_present
    if missing_tk:
        print(f'[TikTok] Warning missing: {missing_tk}')
    else:
        print(f'[OK] tiktok (Tri-modal dataset ready in {tiktok_dest})')
        for fname in sorted(tiktok_req):
            sz = os.path.getsize(os.path.join(tiktok_dest, fname)) / 1024 / 1024
            print(f'       {fname}: {sz:.1f}MB')
else:
    print(f'[Notice] tiktok folder not yet in {DATA_ROOT}. Sẽ tự động nạp từ repo khi chạy.')

print(f'\n[OK] Data preparation verified.')


In [ ]:
# ================================================================
# CELL 3: Pre-flight Check
# [AUDIT FIX v2] Xác nhận yaml configs tồn tại và load đúng
# Hỗ trợ: Baby, Sports, Electronics và TikTok
# ================================================================
import os

STAIR_DIR = '/kaggle/working/STAIR'
DATA_ROOT  = '/kaggle/working/STAIR/data'

# Dataset name -> yaml config path trong STAIR repo
YAML_CONFIGS = {
    'Amazon2014Baby_550_MMRec':        f'{STAIR_DIR}/configs/Amazon2014Baby_550_MMRec.yaml',
    'Amazon2014Sports_550_MMRec':      f'{STAIR_DIR}/configs/Amazon2014Sports_550_MMRec.yaml',
    'Amazon2014Electronics_550_MMRec': f'{STAIR_DIR}/configs/Amazon2014Electronics_550_MMRec.yaml',
    'tiktok':                          f'{STAIR_DIR}/configs/tiktok_MMRec.yaml',
}

print('=== Pre-flight: Checking yaml configs ===')
all_ok = True
for ds, yaml_path in YAML_CONFIGS.items():
    if not os.path.isfile(yaml_path):
        print(f'[MISSING] {yaml_path}')
        all_ok = False
    else:
        with open(yaml_path, 'r') as f:
            content = f.read()
        print(f'[OK] {os.path.basename(yaml_path)}')
        for key in ['monitors', 'which4best', 'epochs', 'batch_size', 'gamma', 'mfiles', 'num_neighbors']:
            for line in content.splitlines():
                if line.strip().startswith(key + ':'):
                    print(f'       {line.strip()}')
                    break

if not all_ok:
    print('\n[WARNING] Some yaml configs missing!')
else:
    print('\n[OK] All yaml configs found. Safe to train.')


In [ ]:
# ================================================================
# CELL 4: Smoke Test (Baby, 1 epoch)
# [AUDIT FIX v2] Dung --config yaml + --root override
# Xac nhan monitors/which4best duoc load dung truoc khi chay full.
# ================================================================
import subprocess, os

STAIR_DIR = '/kaggle/working/STAIR'
DATA_ROOT  = '/kaggle/working/STAIR/data'
os.makedirs('/kaggle/working/logs', exist_ok=True)

print('[Smoke test] Baby, 1 epoch, using --config yaml...')
result = subprocess.run(
    [
        'python', 'main.py',
        '--config', 'configs/Amazon2014Baby_550_MMRec.yaml',
        '--root',   DATA_ROOT,   # only override root, all other params from yaml
        '--epochs', '1',
    ],
    capture_output=True, text=True,
    cwd=STAIR_DIR
)

print('--- stdout (last 60 lines) ---')
lines = result.stdout.splitlines()
print('\n'.join(lines[-60:]))

if result.returncode != 0:
    print('\n--- stderr ---')
    print(result.stderr[-2000:])
    raise RuntimeError('[Smoke test FAILED] Fix errors above before running full training!')

# Verify monitors were loaded
output_full = result.stdout
if 'monitors: []' in output_full:
    raise RuntimeError(
        '[AUDIT ERROR] monitors=[] detected!\n'
        'The yaml config is not being loaded correctly.\n'
        'Check that --config path is valid and freerec version is 0.8.5.'
    )
elif 'RECALL' in output_full.upper() or 'NDCG' in output_full.upper():
    print('\n[OK] Smoke test passed! Metrics (Recall/NDCG) detected in output.')
    print('[OK] Safe to run full 500-epoch training.')
else:
    print('\n[WARNING] Smoke test passed but no Recall/NDCG detected at epoch 1.')
    print('This may be normal if eval-freq > 1 in yaml. Check yaml eval-freq setting.')
    print('[OK] Proceeding to full training.')


In [ ]:
# ================================================================
# CELL 5a: Training — Baby & Sports (DA HOAN THANH)
# [DA COMMENT OUT] 2 tap nay da duoc tai lap thanh cong.
# Log luu tai: /kaggle/working/logs/baby.log
#              /kaggle/working/logs/sports.log
# De chay lai, xoa bo 3 dau ''' o dau va cuoi cell.
# ================================================================

'''
import subprocess, os, time, threading
import pynvml

STAIR_DIR = '/kaggle/working/STAIR'
DATA_ROOT  = '/kaggle/working/STAIR/data'
os.makedirs('/kaggle/working/logs', exist_ok=True)

# VRAM profiler
all_logs = {
    'baby':   {'vram': [], 'time': []},
    'sports': {'vram': [], 'time': []},
}
profiling_active = False
current_ds_key   = None

def hardware_profiler(interval=1.0):
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    t0 = time.time()
    while profiling_active:
        mem = pynvml.nvmlDeviceGetMemoryInfo(handle)
        if current_ds_key:
            all_logs[current_ds_key]['vram'].append(mem.used / 1024**2)
            all_logs[current_ds_key]['time'].append(time.time() - t0)
        time.sleep(interval)
    pynvml.nvmlShutdown()

RUNS_DONE = [
    dict(key='baby',   yaml='configs/Amazon2014Baby_550_MMRec.yaml'),
    dict(key='sports', yaml='configs/Amazon2014Sports_550_MMRec.yaml'),
]

for run in RUNS_DONE:
    key      = run['key']
    log_path = f'/kaggle/working/logs/{key}.log'
    print(f"Training: {key.upper()} | log -> {log_path}")

    current_ds_key   = key
    profiling_active = True
    prof_thread = threading.Thread(target=hardware_profiler, args=(1.0,), daemon=True)
    prof_thread.start()

    t0 = time.time()
    with open(log_path, 'w', encoding='utf-8') as logf:
        result = subprocess.run(
            ['python', 'main.py',
             '--config', run['yaml'],
             '--root',   DATA_ROOT],
            stdout=logf, stderr=subprocess.STDOUT,
            cwd=STAIR_DIR
        )

    elapsed = time.time() - t0
    profiling_active = False
    prof_thread.join(timeout=5)

    if result.returncode != 0:
        print(f'[FAILED] {key.upper()} rc={result.returncode} after {elapsed/60:.1f}min')
        with open(log_path, encoding='utf-8', errors='replace') as f:
            print(''.join(f.readlines()[-30:]))
    else:
        print(f'[OK] {key.upper()} done in {elapsed/60:.1f} min')
        with open(log_path, encoding='utf-8', errors='replace') as f:
            print(''.join(f.readlines()[-15:]))

print('[Done] Baby + Sports training complete!')
'''

print('[SKIP] Cell 5a (Baby & Sports) da duoc comment out.')
print('       Ket qua log co san tai:')
print('       /kaggle/working/logs/baby.log')
print('       /kaggle/working/logs/sports.log')


In [ ]:
# ================================================================
# CELL 5b: Training — Electronics (DANG CHAY)
# Baby + Sports da hoan thanh o Cell 5a (da comment out).
# Cell nay chi chay Electronics voi:
#   - batch_size=4096 (lon nhat trong 3 tap)
#   - gamma=0.4, weight_decay=0.1 (theo Table 4 paper)
# Thoi gian uoc tinh: 4-5h tren T4
# ================================================================
import subprocess, os, time, threading
import pynvml

STAIR_DIR = '/kaggle/working/STAIR'
DATA_ROOT  = '/kaggle/working/STAIR/data'
os.makedirs('/kaggle/working/logs', exist_ok=True)

# VRAM profiler (chi cho electronics)
elec_log = {'vram': [], 'time': []}
profiling_active = False

def hardware_profiler_elec(interval=1.0):
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    t0 = time.time()
    while profiling_active:
        mem = pynvml.nvmlDeviceGetMemoryInfo(handle)
        elec_log['vram'].append(mem.used / 1024**2)
        elec_log['time'].append(time.time() - t0)
        time.sleep(interval)
    pynvml.nvmlShutdown()

log_path = '/kaggle/working/logs/electronics.log'
yaml_cfg = 'configs/Amazon2014Electronics_550_MMRec.yaml'

print('=' * 55)
print('TRAINING: ELECTRONICS')
print(f'Config: {yaml_cfg}')
print(f'Log   : {log_path}')
print('=' * 55)

# Xac nhan yaml ton tai truoc khi bat dau
yaml_full = os.path.join(STAIR_DIR, yaml_cfg)
if not os.path.isfile(yaml_full):
    raise FileNotFoundError(f'Yaml config not found: {yaml_full}')
with open(yaml_full) as f:
    cfg_content = f.read()
print('Yaml config preview:')
for line in cfg_content.splitlines():
    if any(k in line for k in ['monitors', 'which4best', 'epochs', 'batch', 'gamma', 'weight']):
        print(f'  {line.strip()}')
print()

profiling_active = True
prof_thread = threading.Thread(target=hardware_profiler_elec, args=(1.0,), daemon=True)
prof_thread.start()

t0 = time.time()
with open(log_path, 'w', encoding='utf-8') as logf:
    result = subprocess.run(
        [
            'python', 'main.py',
            '--config', yaml_cfg,   # key fix: dung yaml, khong list flag roi
            '--root',   DATA_ROOT,  # chi override root
        ],
        stdout=logf, stderr=subprocess.STDOUT,
        cwd=STAIR_DIR
    )

elapsed = time.time() - t0
profiling_active = False
prof_thread.join(timeout=5)

if result.returncode != 0:
    print(f'[FAILED] Electronics rc={result.returncode} after {elapsed/60:.1f}min')
    print(f'Last 30 lines of {log_path}:')
    with open(log_path, encoding='utf-8', errors='replace') as f:
        print(''.join(f.readlines()[-30:]))
else:
    print(f'[OK] Electronics done in {elapsed/60:.1f} min | log: {log_path}')
    with open(log_path, encoding='utf-8', errors='replace') as f:
        lines = f.readlines()
    print('--- Final metrics (last 20 lines) ---')
    print(''.join(lines[-20:]))

# VRAM summary
if elec_log['vram']:
    print(f'VRAM peak: {max(elec_log["vram"]):.0f} MB')
    print(f'VRAM avg : {sum(elec_log["vram"])/len(elec_log["vram"]):.0f} MB')


In [ ]:
# ================================================================
# CELL 5c: Training — TikTok (Micro-video Multimodal Recommendation)
# - Dataset: 9,308 Users x 6,710 Items (59,541 train interactions)
# - Scenario: Content-driven micro-video recommendation
# - Modalities: Tri-modal (Vision: 128-d, Text: 768-d, Audio: 128-d)
# - Config: configs/tiktok_MMRec.yaml (gamma=0.05, num_neighbors='3-3-3')
# - Expected runtime: ~15-25 min on GPU T4 / P100 (super lightweight)
# ================================================================
import subprocess, os, time, threading, shutil
import pynvml

STAIR_DIR = '/kaggle/working/STAIR'
DATA_ROOT  = '/kaggle/working/STAIR/data'
os.makedirs('/kaggle/working/logs', exist_ok=True)

# VRAM profiler cho TikTok
all_logs = globals().get('all_logs', {})
all_logs['tiktok'] = {'vram': [], 'time': []}
tiktok_log = all_logs['tiktok']
profiling_active = False

def hardware_profiler_tiktok(interval=1.0):
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    t0 = time.time()
    while profiling_active:
        mem = pynvml.nvmlDeviceGetMemoryInfo(handle)
        tiktok_log['vram'].append(mem.used / 1024**2)
        tiktok_log['time'].append(time.time() - t0)
        time.sleep(interval)
    pynvml.nvmlShutdown()

log_path = '/kaggle/working/logs/tiktok.log'
yaml_cfg = 'configs/tiktok_MMRec.yaml'

print('=' * 65)
print('TRAINING: STAIR BASELINE ON TIKTOK DATASET (MICRO-VIDEO)')
print(f'Config: {yaml_cfg}')
print(f'Log   : {log_path}')
print('=' * 65)

# Xác nhận yaml config tồn tại
yaml_full = os.path.join(STAIR_DIR, yaml_cfg)
if not os.path.isfile(yaml_full):
    raise FileNotFoundError(f'Yaml config not found: {yaml_full}')
with open(yaml_full) as f:
    cfg_content = f.read()
print('Yaml config preview:')
for line in cfg_content.splitlines():
    if any(k in line for k in ['monitors', 'which4best', 'epochs', 'batch', 'gamma', 'mfiles', 'num_neighbors']):
        print(f'  {line.strip()}')
print()

# Đảm bảo dữ liệu tiktok nằm trong DATA_ROOT
tiktok_data = os.path.join(DATA_ROOT, 'tiktok')
if not os.path.exists(tiktok_data) or not os.path.exists(os.path.join(tiktok_data, 'train.txt')):
    repo_tiktok = os.path.join(STAIR_DIR, 'data', 'tiktok')
    if os.path.exists(repo_tiktok):
        shutil.copytree(repo_tiktok, tiktok_data, dirs_exist_ok=True)
        print(f'[OK] Copied TikTok dataset from repo to {tiktok_data}')

profiling_active = True
prof_thread = threading.Thread(target=hardware_profiler_tiktok, args=(1.0,), daemon=True)
prof_thread.start()

t0 = time.time()
with open(log_path, 'w', encoding='utf-8') as logf:
    result = subprocess.run(
        [
            'python', 'main.py',
            '--config', yaml_cfg,
            '--root',   DATA_ROOT,
        ],
        stdout=logf, stderr=subprocess.STDOUT,
        cwd=STAIR_DIR
    )

elapsed = time.time() - t0
profiling_active = False
prof_thread.join(timeout=5)

if result.returncode != 0:
    print(f'[FAILED] TikTok rc={result.returncode} after {elapsed/60:.1f}min')
    print(f'Last 30 lines of {log_path}:')
    with open(log_path, encoding='utf-8', errors='replace') as f:
        print(''.join(f.readlines()[-30:]))
else:
    print(f'[OK] TikTok training complete in {elapsed/60:.1f} min | log: {log_path}')
    with open(log_path, encoding='utf-8', errors='replace') as f:
        lines = f.readlines()
    print('--- Final metrics (last 20 lines) ---')
    print(''.join(lines[-20:]))

# VRAM summary
if tiktok_log['vram']:
    print(f'VRAM peak: {max(tiktok_log["vram"]):.0f} MB')
    print(f'VRAM avg : {sum(tiktok_log["vram"])/len(tiktok_log["vram"]):.0f} MB')


In [ ]:
# ================================================================
# CELL 6: Extract Final Metrics & Compare with Benchmarks
# Parse VALID/TEST lines from log -> extract Recall@10/20, NDCG@10/20
# Hiển thị bảng đối chuẩn cho cả 4 tập dữ liệu: Baby, Sports, Electronics, TikTok
# ================================================================
import re, os, json

# Reference values (Baseline tái lập chuẩn xác)
PAPER = {
    'baby':        dict(r10=0.0674, r20=0.1042, n10=0.0359, n20=0.0454),
    'sports':      dict(r10=0.0743, r20=0.1111, n10=0.0405, n20=0.0500),
    'electronics': dict(r10=0.0440, r20=0.0663, n10=0.0245, n20=0.0302),
    'tiktok':      dict(r10=0.0000, r20=0.0000, n10=0.0000, n20=0.0000),  # Mốc tham chiếu thực nghiệm mới
}

def parse_metrics_from_log(log_path):
    """Parse best TEST Recall@10/20, NDCG@10/20 from freerec log."""
    if not os.path.exists(log_path):
        return None, 'Log file not found'

    with open(log_path, encoding='utf-8', errors='replace') as f:
        lines = f.readlines()

    # Check if monitors was set
    full_text = ''.join(lines)
    if '[monitors: []]' in full_text:
        return None, 'monitors=[] detected — yaml config not loaded correctly!'

    # Find the best-epoch TEST line
    test_lines = [l for l in lines if 'TEST' in l and ('RECALL' in l.upper() or 'NDCG' in l.upper())]
    if not test_lines:
        test_lines = [l for l in lines if 'VALID' in l and ('RECALL' in l.upper() or 'NDCG' in l.upper())]
        if not test_lines:
            return None, 'No RECALL/NDCG lines found in log'

    last = test_lines[-1]
    r10 = re.search(r'RECALL@10[^:]*:\s*([0-9.]+)', last, re.IGNORECASE)
    r20 = re.search(r'RECALL@20[^:]*:\s*([0-9.]+)', last, re.IGNORECASE)
    n10 = re.search(r'NDCG@10[^:]*:\s*([0-9.]+)', last, re.IGNORECASE)
    n20 = re.search(r'NDCG@20[^:]*:\s*([0-9.]+)', last, re.IGNORECASE)

    if not all([r10, r20, n10, n20]):
        return None, f'Could not parse all metrics from: {last.strip()}'

    return dict(
        r10=float(r10.group(1)), r20=float(r20.group(1)),
        n10=float(n10.group(1)), n20=float(n20.group(1))
    ), 'OK'

print('=' * 72)
print('STAIR Baseline Benchmark Results — Across All 4 Datasets')
print('=' * 72)
print(f'{"Dataset":<14} {"Metric":<12} {"Reference":>10} {"Reproduced":>12} {"Status/Delta":>14}')
print('-' * 66)

summary = {}
for key in ['baby', 'sports', 'electronics', 'tiktok']:
    log_path = f'/kaggle/working/logs/{key}.log'
    repro, status = parse_metrics_from_log(log_path)

    if repro is None:
        print(f'{key:<14} [STATUS] {status}')
        continue

    summary[key] = repro
    paper = PAPER[key]

    for label, r_key in [('Recall@10', 'r10'), ('Recall@20', 'r20'), ('NDCG@10', 'n10'), ('NDCG@20', 'n20')]:
        p_val = paper[r_key]
        r_val = repro[r_key]
        if p_val and p_val > 0:
            delta = (r_val - p_val) / p_val * 100
            flag = '[>5%!]' if abs(delta) > 5 else '[ok] '
            print(f'{key:<14} {label:<12} {p_val:>10.4f} {r_val:>12.4f} {delta:>+10.2f}% {flag}')
        else:
            print(f'{key:<14} {label:<12} {"N/A":>10} {r_val:>12.4f} {"[NEW BENCH]":>14}')

    print('-' * 66)

# Save summary
with open('/kaggle/working/reproduction_summary.json', 'w') as f:
    json.dump({'reference': PAPER, 'results': summary}, f, indent=2)
print('\n[Saved] /kaggle/working/reproduction_summary.json')


In [ ]:
# ================================================================
# CELL 7: VRAM Usage Visualisation
# Vẽ biểu đồ tiêu thụ bộ nhớ GPU trên cả 4 tập dữ liệu
# ================================================================
import matplotlib.pyplot as plt

KEYS   = ['baby', 'sports', 'electronics', 'tiktok']
COLORS = ['#1f77b4', '#ff7f0e', '#d62728', '#2ca02c']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('GPU VRAM Usage — STAIR Reproduction & TikTok Benchmark (Kaggle T4)',
             fontsize=15, fontweight='bold')

all_logs = globals().get('all_logs', {})
for idx, (key, color) in enumerate(zip(KEYS, COLORS)):
    ax = axes[idx // 2, idx % 2]
    if key in all_logs and all_logs[key]['vram']:
        times = all_logs[key]['time']
        vrams = all_logs[key]['vram']
        ax.plot(times, vrams, color=color, linewidth=1.5)
        ax.fill_between(times, vrams, color=color, alpha=0.25)
        ax.axhline(max(vrams), color='black', linestyle='--', linewidth=0.8,
                   label=f'Peak: {max(vrams):.0f} MB')
        ax.legend(fontsize=9)
    else:
        ax.text(0.5, 0.5, f'No live profiler data for {key}\n(Check log file: logs/{key}.log)',
                ha='center', va='center', transform=ax.transAxes, color='gray')
    ax.set_title(key.capitalize(), fontsize=12)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('VRAM (MB)')
    ax.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout(rect=[0, 0.02, 1, 0.95])
plt.savefig('/kaggle/working/vram_profile.png', dpi=150, bbox_inches='tight')
plt.show()
print('[Saved] /kaggle/working/vram_profile.png')
